In [1]:
from Bio import motifs
import Bio.SeqUtils.lcc as lcc
import numpy as np
import pandas as pd
import os
from time import time

In [2]:
base_pairs = {'A': 'T', 'T': 'A', 'G': 'C', 'C': 'G'}

In [3]:
def reverse_complement(x):
    return str(''.join(base_pairs.get(base, base) for base in reversed(x)))

In [4]:
base_path = '/tamir2/shaicohen1/CRISPR_MAAGAD/CRISPR_review/code/methylation_LCC_code/'

In [5]:
with open(os.path.join(base_path, 'topEnriched.313.meme.txt'), 'r') as handle:
    motifs_methylation = motifs.parse(handle, "minimal")
pwm_methylation = [x.counts.normalize(pseudocounts={'A':0.295, 'C': 0.205, 'G': 0.205, 'T': 0.295}) for x in motifs_methylation]
pssm_methylation = [x.log_odds(motifs_methylation.background) for x in pwm_methylation]

In [6]:
def calc_methyl_vector(motifs_methylation, row):
    seq = row.loc['target_seq'][:20]
    pre_seq = row.loc['up_seq']
    post_seq = row.loc['down_seq']

    if type(seq) == float:
        return row

    len_seq = len(seq)

    forward_seq = seq+post_seq[:15]
    backward_seq = reverse_complement(seq)+ reverse_complement(pre_seq[-15:])


    results_arr = motifs_methylation[0].pssm.calculate(backward_seq)[:len_seq].reshape(1, -1)
    for motif in motifs_methylation:
        np.append(results_arr, motif.pssm.calculate(backward_seq)[:len_seq].reshape(1, -1), axis=0)

    backward_vec = np.fliplr(np.amax(results_arr, axis=0).reshape(1, -1))


    results_arr = motifs_methylation[0].pssm.calculate(forward_seq)[:len_seq].reshape(1, -1)
    for motif in motifs_methylation:
        np.append(results_arr, motif.pssm.calculate(forward_seq)[:len_seq].reshape(1, -1), axis =0)

    np.append(results_arr, backward_vec, axis =0)

    max_vec = np.amax(results_arr, axis=0)

    for ii in range(len_seq):
        row[f'methylation_{ii}'] = round(max_vec[ii], 4)

    row[f'methylation_max'] = round(np.max(max_vec), 4)
    row[f'methylation_mean'] = round(np.mean(max_vec), 4)
    row[f'methylation_std'] = round(np.std(max_vec), 4)

    return row

In [7]:
def lcc_features(row):
    if type(row.loc['target_seq']) == float:
        return row
    curr_seq = row.loc['up_seq'][-10:]+row.loc['target_seq'][:20]+row.loc['down_seq'][:10]
    lcc_features = lcc.lcc_mult(curr_seq, 10)

    for ii, curr_lcc in enumerate(lcc_features):
        row[f'LCC_{ii}'] = round(curr_lcc, 4)
    return row


In [8]:
# path_data = '/tamir2/shaicohen1/CRISPR_MAAGAD/CRISPR_review/code/methylation_LCC_code/output_sites/fix_human/'
# list_files = os.listdir(path_data)
# list_files = [x for x in list_files if '.csv' in x]
curr_df = pd.read_csv('/tamir2/shaicohen1/CRISPR_MAAGAD/CRISPR_review/code/final_feats/Tomato-hairy-roots_eff.csv')
pre_df = curr_df.copy()
# for filename in list_files:
for i in [0]:
    a=time()
    # print(f'starting {filename}')
    # curr_df = pd.read_csv(os.path.join(path_data, filename))
    cols_to_drop = [x for x in curr_df.columns if 'methylation' in x]
    cols_to_drop.extend([x for x in curr_df.columns if 'LCC' in x])
    curr_df = curr_df.drop(columns=cols_to_drop)
    print([x for x in curr_df.columns if 'methylation' in x])
    curr_df = curr_df.apply(lambda row: calc_methyl_vector(motifs_methylation, row), axis = 1)
    curr_df = curr_df.apply(lambda row: lcc_features(row), axis = 1)
    # curr_df.to_csv(path_data+'output/'+filename[:-4]+'_lcc_meth.csv',index=False)
    # curr_df.to_csv('/tamir2/shaicohen1/CRISPR_MAAGAD/CRISPR_review/code/final_feats/_Tomato-hairy-roots_eff.csv',index=False)

    # print(filename,time()-a)
print('done')
#
# df_testing_new = df_testing_new.apply(lambda row: calc_methyl_vector(motifs_methylation, row), axis = 1)
#df_training = df_training.apply(lambda row: lcc_features(row), axis = 1)

[]
done


In [13]:
pd.testing.assert_frame_equal(
    curr_df[cols_to_drop],
    pre_df[cols_to_drop],
    rtol=0,
    atol=0.0001)
# curr_df[cols_to_drop]==pre_df[cols_to_drop]

In [15]:
# path_data = '/tamir2/shaicohen1/CRISPR_MAAGAD/CRISPR_review/code/methylation_LCC_code/output_sites/fix_human/'
# list_files = os.listdir(path_data)
# list_files = [x for x in list_files if '.csv' in x]
# for f in list_files:
#     df = pd.read_csv(path_data+f)
#     print(np.all(df.target_seq.apply(lambda x: len(x)==23)))

True
True
True
True
True


/var/tmp/pbs.2520224.power9.tau.ac.il/ipykernel_78836/3203523950.py:5: DtypeWarning: Columns (541,542,543,544,545,546,547,548,549,550,551,552,553,554,555,556,557,558) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path_data+f)
/var/tmp/pbs.2520224.power9.tau.ac.il/ipykernel_78836/3203523950.py:5: DtypeWarning: Columns (546,547,548,549,550,551,552,553,554,555,556,557,558,559,560,561,562,563) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path_data+f)


True
True


/var/tmp/pbs.2520224.power9.tau.ac.il/ipykernel_78836/3203523950.py:5: DtypeWarning: Columns (776,777,778,779,780,781,782,783,784,785,786,787,788,789,790,791,792,793) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path_data+f)


True
